# 01 — Data pipeline + QLoRA SFT

End-to-end Kaggle path for **Phase 1** (public fixtures → grounded pairs → versioned splits) and **Phase 2** (Unsloth QLoRA SFT).

| Stage | Default | GPU needed? |
|-------|---------|-------------|
| Ingest → chunk → generate → filter → select | Always runs (offline fixtures) | No |
| Train dry-run (`sft_plan.json`) | Always runs | No |
| Real train (`RUN_TRAIN=True`) | **Off** | Yes (T4) |

**Seed:** `3407`. **Primary model:** `unsloth/Llama-3.2-3B-Instruct` (4-bit).

Adapter output: `outputs/adapters/llama32-3b-ecra-sft/`

Set `RUN_TRAIN = True` only after the data cells succeed and a T4 GPU is attached.

## 0. Knobs

In [ ]:
# --- User knobs (edit these) ---
RUN_TRAIN = False          # True = Unsloth QLoRA train (needs GPU)
MAX_STEPS = 20             # smoke train; set None for full epoch when ready
MAX_SAMPLES = 5            # fixture samples per public source
DOWNLOAD_HF = False        # True streams a tiny HF sample (optional)
USE_LLM_JUDGE = False      # proxy judge only when True (no extra GPU bill)
CONFIG_PATH = "configs/default.yaml"   # or configs/llama32-8b.yaml

print("RUN_TRAIN=", RUN_TRAIN, "MAX_STEPS=", MAX_STEPS)

## 1. Repo path + installs

Clone once into `/kaggle/working` if needed. This cell adds **`src/`** (not the repo root) to `sys.path` so `import earnings_call_research_assistant` works.

In [ ]:
from pathlib import Path
import os
import sys

IN_KAGGLE = Path("/kaggle").exists()
print("IN_KAGGLE:", IN_KAGGLE)

# Prefer an existing clone; otherwise clone.
if IN_KAGGLE:
    work = Path("/kaggle/working")
    repo = work / "earnings-call-research-assistant"
    if not (repo / "src" / "earnings_call_research_assistant" / "inference.py").exists():
        %cd /kaggle/working
        !git clone --depth 1 https://github.com/nuwanda94/earnings-call-research-assistant.git
        repo = work / "earnings-call-research-assistant"
    REPO = repo.resolve()
else:
    # Local: notebook lives under notebooks/
    REPO = Path("..").resolve()
    if not (REPO / "src" / "earnings_call_research_assistant").is_dir():
        REPO = Path.cwd().resolve()

SRC = REPO / "src"
assert (SRC / "earnings_call_research_assistant" / "inference.py").exists(), SRC

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
os.chdir(REPO)
print("REPO:", REPO)
print("cwd:", Path.cwd())

import earnings_call_research_assistant as ecra
print("package:", ecra.__file__, "v", getattr(ecra, "__version__", "?"))

In [ ]:
if IN_KAGGLE:
    # Data path is pure Python; train path needs Unsloth + TRL.
    %pip install -q pyyaml
    if RUN_TRAIN:
        %pip install -q unsloth transformers accelerate bitsandbytes datasets trl peft

## 2. Phase 1 — public ingest (offline fixtures)

In [ ]:
from earnings_call_research_assistant.data import (
    ingest_catalog,
    list_sources,
    write_jsonl,
)

print("Catalog:")
for s in list_sources():
    print(f"  - {s.source_id}: {s.display_name} ({s.role})")

RAW = Path("data/raw/public_sample.jsonl")
records = ingest_catalog(max_samples=MAX_SAMPLES, download=DOWNLOAD_HF)
write_jsonl(records, RAW)
print(f"Wrote {len(records)} records -> {RAW.resolve()}")
print(records[0].to_dict() if records else "(empty)")

## 3. Chunking + proposition extraction

In [ ]:
from earnings_call_research_assistant.data import ChunkConfig, chunk_records, write_chunks_jsonl

CHUNKS = Path("data/processed/chunks.jsonl")
chunk_cfg = ChunkConfig(window_sentences=4, stride_sentences=2)
chunks = chunk_records(records, config=chunk_cfg)
write_chunks_jsonl(chunks, CHUNKS)
n_props = sum(len(c.propositions) for c in chunks)
print(f"chunks={len(chunks)} propositions={n_props} -> {CHUNKS}")
if chunks:
    print("sample props:", [p.text for p in chunks[0].propositions[:3]])

## 4. Grounded synthetic Q&A / summary pairs (templates)

In [ ]:
from earnings_call_research_assistant.data import GenerateConfig, generate_pairs, write_pairs_jsonl

PAIRS = Path("data/processed/grounded_pairs.jsonl")
gen_cfg = GenerateConfig(max_qa_per_chunk=2, include_summary=True, use_llm=False)
pairs = generate_pairs(chunks, config=gen_cfg)
write_pairs_jsonl(pairs, PAIRS)
n_qa = sum(1 for p in pairs if p.task == "qa")
n_sum = sum(1 for p in pairs if p.task == "summary")
print(f"pairs={len(pairs)} qa={n_qa} summary={n_sum} -> {PAIRS}")
if pairs:
    p0 = pairs[0]
    print("task:", p0.task)
    print("instruction:", p0.instruction[:200])
    print("output:", p0.output[:200])

## 5. Multi-stage filter (heuristic → dedup → optional judge)

In [ ]:
from earnings_call_research_assistant.data import FilterConfig, filter_pairs, write_filter_report

FILTERED = Path("data/processed/filtered_pairs.jsonl")
REPORT = Path("data/processed/filter_report.json")
filt_cfg = FilterConfig(
    min_output_chars=40,
    near_dup_jaccard=0.88,
    use_llm_judge=USE_LLM_JUDGE,
    min_judge_score=0.6,
)
kept, report = filter_pairs(pairs, config=filt_cfg)
write_pairs_jsonl(kept, FILTERED)
write_filter_report(report, REPORT)
print(
    f"in={report.n_in} kept={report.n_kept} "
    f"dropped={report.n_in - report.n_kept} by_stage={report.dropped_by_stage}"
)
print("->", FILTERED)

## 6. Diversity selection + versioned splits

Writes `data/processed/ecra-sft-v0.1.0/{train,val,test}.jsonl` + `manifest.json`.

In [ ]:
from earnings_call_research_assistant.data import (
    DATASET_VERSION,
    SelectConfig,
    select_and_split,
    write_splits,
)

OUT_DIR = Path("data/processed") / DATASET_VERSION
sel_cfg = SelectConfig(
    target_min=1,
    target_max=6000,
    max_per_source=2500,
    diversity_jaccard_cap=0.72,
    seed=94,
    dataset_version=DATASET_VERSION,
)
splits, sel_report = select_and_split(kept, config=sel_cfg)
paths = write_splits(splits, OUT_DIR, report=sel_report, config=sel_cfg)
print(
    f"version={sel_report.dataset_version} selected={sel_report.n_selected} "
    f"train={sel_report.n_train} val={sel_report.n_val} test={sel_report.n_test}"
)
print("by_source=", sel_report.by_source)
print("by_task=", sel_report.by_task)
for k, v in paths.items():
    print(f"  {k}: {v}")

## 7. Phase 2 — SFT dry-run (always)

Formats chat examples and writes `outputs/sft_plan.json`. **Does not** load weights.

In [ ]:
from earnings_call_research_assistant.training.sft import run_sft
import json

plan = run_sft(
    config_path=CONFIG_PATH,
    dataset_dir=OUT_DIR,
    dry_run=True,
    max_steps=MAX_STEPS,
    require_train=False,
)
print(
    f"dry_run={plan.dry_run} model={plan.model_name} seed={plan.seed} "
    f"train={plan.n_train} val={plan.n_val}"
)
print("adapter_dir:", plan.adapter_dir)
print("plan file: outputs/sft_plan.json")
print(json.dumps(plan.to_dict(), indent=2)[:1800])

## 8. Real QLoRA train (optional)

Only runs when `RUN_TRAIN = True` in the knobs cell. Needs a **T4 GPU** and Unsloth installed.

After success, copy `outputs/adapters/llama32-3b-ecra-sft/` off the session before it dies.

In [ ]:
import torch

print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

if not RUN_TRAIN:
    print("Skipped train (RUN_TRAIN=False). Set RUN_TRAIN=True and re-run this cell on GPU.")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("RUN_TRAIN=True but no CUDA GPU. Enable T4 in Kaggle settings.")
    train_plan = run_sft(
        config_path=CONFIG_PATH,
        dataset_dir=OUT_DIR,
        dry_run=False,
        max_steps=MAX_STEPS,
        require_train=True,
    )
    print("Train finished.")
    print("adapter:", train_plan.adapter_dir)
    print("notes:", train_plan.notes[-3:])

## 9. Smoke-generate with the adapter (optional)

Runs only if `RUN_TRAIN` succeeded and the adapter directory exists.

In [ ]:
from earnings_call_research_assistant.inference import InferenceConfig, InferenceHarness
import yaml

adapter = Path("outputs/adapters/llama32-3b-ecra-sft")
if not RUN_TRAIN or not adapter.exists():
    print("Skip adapter smoke (no adapter on disk yet).")
else:
    with Path(CONFIG_PATH).open() as f:
        cfg = InferenceConfig.from_mapping(yaml.safe_load(f))
    # Load base; attach LoRA if Unsloth PEFT dir is present.
    harness = InferenceHarness.from_pretrained(cfg)
    try:
        harness.model.load_adapter(str(adapter))
        print("Loaded adapter from", adapter)
    except Exception as e:
        print("Could not load_adapter (base-only smoke):", e)
    q = "Summarize prepared remarks vs Q&A on a US large-cap earnings call."
    print(harness.generate(q))

## Done

Checklist:

1. `data/processed/ecra-sft-v0.1.0/train.jsonl` exists
2. `outputs/sft_plan.json` shows `seed: 3407`
3. (Optional) adapter under `outputs/adapters/llama32-3b-ecra-sft/`

Next: `notebooks/00_baseline_inference.ipynb` for base snapshots, then `scripts/eval_research_panel.py --run --adapter-dir ...` on GPU.

See `docs/REPRODUCIBILITY.md` and `docs/DATA_CARD.md`.